In [2]:
# !uv pip install ipykernel

# Django Shell을 주피터랩에서 실행할 수 있도록 환경설정
import os
import django
os.environ['DJANGO_SETTINGS_MODULE'] = "config.settings"
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = "true"

django.setup()

In [4]:
from polls.models import Question, Choice

# ModelClass.objects : (Model)Manager객체 -> select 처리 메소드 제공
type(Question.objects)

django.db.models.manager.Manager

In [5]:
#PK=1인 질문 조회
result = Question.objects.get(pk=1)
print(type(result))

<class 'polls.models.Question'>


In [6]:
result

<Question: 1. 좋아하는 색은 무엇인가요?>

In [7]:
result.pk, result.question_text, result.pub_date

(1,
 '좋아하는 색은 무엇인가요?',
 datetime.datetime(2026, 7, 24, 6, 39, 37, 399426, tzinfo=datetime.timezone.utc))

# 조회
- Model Manager의 메소드를 이용해서 조회
- `all()` : 전체 조회
- (where절) 조건으로 조회
    - `filter(조건)` : 조건이 true인 행 조회
    - `exclude(조건)` : 조건이 false인 행 조회
    - `get(조건)` : 조건이 true인 한 개 행 조회(조회 결과가 하나일 때 사용)
- all, filter, exclude, get 반환 타입 : QuerySet
- get 반환 : 모델 객체

In [11]:
# 모든 질문 조회
result = Question.objects.all()
print(type(result))
print("결과 개수 :", len(result))
print("첫 번째 조회 결과 :", result[0], type(result[0]))

<class 'django.db.models.query.QuerySet'>
결과 개수 : 4
첫 번째 조회 결과 : 1. 좋아하는 색은 무엇인가요? <class 'polls.models.Question'>


In [13]:
for question in result:
    print(question.pk, question.question_text, question.pub_date)

1 좋아하는 색은 무엇인가요? 2026-07-24 06:39:37.399426+00:00
2 싫어하는 색은 무엇인가요? 2026-07-24 06:40:22.024140+00:00
3 좋아하는 동물은 무엇인가요? 2026-07-24 06:40:38.071426+00:00
4 가고 싶은 여행지는 어디인가요? 2026-07-24 06:40:56.399515+00:00


In [ ]:
result[:2]  # list
# result[-1] # 음수 indexing 지원 X

[<Question: 1. 좋아하는 색은 무엇인가요?>, <Question: 2. 싫어하는 색은 무엇인가요?>]

In [15]:
result.first()  # 첫 번째 값
result.last()  # 마지막 값

<Question: 4. 가고 싶은 여행지는 어디인가요?>

In [16]:
Question.objects.get(pk=1)  # where pk=1

<Question: 1. 좋아하는 색은 무엇인가요?>

In [29]:
# Choice.objects.get(votes=0)  # 조회 결과가 여러 개인 경우 에러 발생(MultipleObjectsReturned) -> PK 조회 시 사용
try:
    Choice.objects.get(votes=300000)  # 조회 결과가 없는 경우 에러 발생(DoesNotExist) -> try except 사용
except:
    print("조회 결과가 없습니다.")

조회 결과가 없습니다.


In [22]:
# 조회 결과가 여러 개인 경우
result = Choice.objects.filter(votes=0) # QuerySet
len(result)

11

In [24]:
result[0].pk, result[0].choice_text, result[0].votes

(2, '검정색', 0)

In [25]:
result[0].question

<Question: 1. 좋아하는 색은 무엇인가요?>

In [26]:
result = Choice.objects.exclude(votes=0) # where not vote=0
len(result)

5

In [27]:
for c in result:
    print(c.choice_text, c.votes)

파랑색 12
리스본 8
포르투갈 12
싱가포르 3
영국 1


In [54]:
# Where 조건
# Field명__연산자 = 비교할 값
result = Choice.objects.filter(votes=0)  # where vote = 0
result = Choice.objects.filter(votes__lt=50)  # where vote < 50
result = Choice.objects.filter(votes__lte=60)  # vote <= 60
result = Choice.objects.filter(votes__gt=50)  # vote > 50 

# 문자열
result = Choice.objects.filter(choice_text="빨간색")
result = Choice.objects.filter(choice_text__startswith="파랑") # choice_text like '파랑%'
result = Choice.objects.filter(choice_text__endswith="색") # choice_text like '%색'
result = Choice.objects.filter(choice_text__contains="라") # choice_text like '%라%'

result = Choice.objects.filter(choice_text__in = ['보라색', '빨간색', '강아지'])
result = Choice.objects.filter(votes__range = [50, 200])  # vote between 50 and 100

for choice in result:
    print(choice.choice_text, choice.votes)

print(result.query)  # QuerySet.query : 실행된 SQL문 

고양이 160
강아지 120
SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" WHERE "polls_choice"."votes" BETWEEN 50 AND 200


In [ ]:
# 조건이 여러 개인 경우 - AND, OR
# And - 조건 나열
result = Choice.objects.filter(
    votes__lt = 100,
    choice_text__in = ['파랑색', '검정색', '강아지', '고양이']
)
print(result.query)

for choice in result:
    print(choice.choice_text, choice.votes)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."choice_text" IN (파랑색, 검정색, 강아지, 고양이) AND "polls_choice"."votes" < 100)
파랑색 12
검정색 0
파랑색 0


In [ ]:
# OR - Q 클래스에 조건 넣어주고 `|` 연산으로 묶어줌
## Q(조건) | Q(조건) | Q(조건)
## ~Q(조건) : not 조건
from django.db.models import Q

# vote가 10 이하거나 100 이상
result = Choice.objects.filter(
    Q(votes__lte=10) | Q(votes__gte=100)
)

print(result.query)
for choice in result:
    print(choice.choice_text, choice.votes)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."votes" <= 10 OR "polls_choice"."votes" >= 100)
검정색 0
보라색 0
초록색 0
파랑색 0
노란색 367
빨간색 5
분홍색 0
고양이 160
강아지 120
기린 0
호랑이 0
리스본 8
싱가포르 3
영국 1


In [ ]:
# 결과 정렬 - order_by("기준 컬럼") : ASC 정렬, DESC 정렬("-기준 컬럼")
## votes의 오름차순
result = Choice.objects.all().order_by("votes")

## 내림차순
result = Choice.objects.all().order_by("-votes")

## 2(n)차 정렬
### votes: DESC, choice_text: ASC
result = Choice.objects.all().order_by("-votes", "choice_text")

print(result.query)
for choice in result:
    print(choice.choice_text, choice.votes)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" ORDER BY "polls_choice"."votes" DESC, "polls_choice"."choice_text" ASC
노란색 367
고양이 160
강아지 120
파랑색 12
포르투갈 12
리스본 8
빨간색 5
싱가포르 3
영국 1
검정색 0
기린 0
보라색 0
분홍색 0
초록색 0
파랑색 0
호랑이 0


In [72]:
# 특정 컬럼들만 지정해서 조회 - select 컬럼명

result = Choice.objects.all().values("pk", "choice_text")

print(type(result))  # QuerySet
print(result.query)
print(type(result[0]))  # 개별 결과: dict

for value in result:
    print(value['pk'], value['choice_text'], value)

<class 'django.db.models.query.QuerySet'>
SELECT "polls_choice"."id" AS "pk", "polls_choice"."choice_text" AS "choice_text" FROM "polls_choice"
<class 'dict'>
1 파랑색 {'pk': 1, 'choice_text': '파랑색'}
2 검정색 {'pk': 2, 'choice_text': '검정색'}
3 보라색 {'pk': 3, 'choice_text': '보라색'}
4 초록색 {'pk': 4, 'choice_text': '초록색'}
5 파랑색 {'pk': 5, 'choice_text': '파랑색'}
6 노란색 {'pk': 6, 'choice_text': '노란색'}
7 빨간색 {'pk': 7, 'choice_text': '빨간색'}
8 분홍색 {'pk': 8, 'choice_text': '분홍색'}
9 고양이 {'pk': 9, 'choice_text': '고양이'}
10 강아지 {'pk': 10, 'choice_text': '강아지'}
11 기린 {'pk': 11, 'choice_text': '기린'}
12 호랑이 {'pk': 12, 'choice_text': '호랑이'}
13 리스본 {'pk': 13, 'choice_text': '리스본'}
14 포르투갈 {'pk': 14, 'choice_text': '포르투갈'}
15 싱가포르 {'pk': 15, 'choice_text': '싱가포르'}
16 영국 {'pk': 16, 'choice_text': '영국'}
